# Equipo 19 · Avance 2: Endo-STTN limpia los frames → ajuste fino del estimador de profundidad

| Paso | Script | Qué hace |
|---|---|---|
| 1 | `codigo/preparar_datos.py` | Extrae de C3VDv2 frames, **profundidad real** y máscara de reflejos (320×256) y divide por video |
| 2 | `codigo/limpiar_endosttn.py` | **Endo-STTN congelado** borra los reflejos → `rgb_limpio` |
| 3 | `codigo/ajustar_profundidad.py` | Ajuste fino de **AF-SfMLearner** con la profundidad real: modelo C (frames originales) y D (frames limpios) |
| 4 | `codigo/evaluar_profundidad.py` | Compara A, B, C y D contra la profundidad real en videos no vistos |

**Hipótesis:** el estimador ajustado con frames limpios (D) estima mejor la profundidad que el ajustado con
frames originales (C), sobre todo donde había reflejos.

> Versión **local**. La versión para Google Colab es `Equipo19_Avance2_colab.ipynb`.

## 0. Configuración

Antes: `pip install -r requirements.txt`, `python preparar_entorno.py` (AF-SfMLearner, sus pesos y el
preentrenado de Endo-STTN) y los ZIPs de C3VDv2 en `datos/raw/`. Abre el notebook dentro del repositorio.
Los pasos 1–3 tardan; puedes correrlos aquí (`CORRER_* = True`) o en la terminal y usar el notebook para ver resultados.

In [ ]:
%matplotlib inline
from pathlib import Path
import os, sys, json, subprocess

# ---------- Ajusta si hace falta ----------
CORRER_PREPARAR = False   # paso 1 (minutos; una sola vez)
CORRER_LIMPIEZA = False   # paso 2 (minutos con GPU; una sola vez)
CORRER_AJUSTE   = False   # paso 3 (dos entrenamientos: C y D)
CORRER_EVALUAR  = True    # paso 4
EPOCAS, SEMILLA = 15, 0
# ------------------------------------------

def buscar_repo():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "codigo" / "c3vd.py").exists():
            return base.resolve()
    raise FileNotFoundError("Abre el notebook dentro del repositorio (no encuentro codigo/c3vd.py)")

REPO = buscar_repo()

In [ ]:
os.chdir(REPO)
sys.path.insert(0, str(REPO / "codigo"))
import numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
import c3vd, modelo_profundidad as mp

def correr(*args):
    r = subprocess.run([sys.executable, *map(str, args)])
    assert r.returncode == 0, f"falló: {args}"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Repositorio :", REPO)
print("Dispositivo :", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "(sin GPU: el ajuste fino será lento)")
print("ZIPs C3VDv2 :", len(list(c3vd.DIR_RAW.glob("*.zip"))), "en", c3vd.DIR_RAW)

## 1. Datos: frames, profundidad real y reflejos

C3VDv2 trae, para cada frame de color, su **profundidad real** (obtenida al registrar el video con el modelo 3D
del colon). Se submuestrea 1 de cada 5 frames, se reduce a 320×256 y se calcula la máscara de reflejos por umbral
de brillo. La división es **por video**: los videos de prueba nunca se ven al entrenar.

In [ ]:
if CORRER_PREPARAR:
    correr("codigo/probar_formato.py")
    correr("codigo/preparar_datos.py")
    if "DRIVE" in globals():   # Colab: guardar lo preparado para la próxima vez
        shutil.make_archive(str(DRIVE / "c3vd_preparado"), "zip", root_dir=REPO / "datos", base_dir="c3vd")

assert (c3vd.DIR_DATOS / "division.json").exists(), "Faltan los datos preparados: pon CORRER_PREPARAR = True o corre  python codigo/preparar_datos.py"
division = json.load(open(c3vd.DIR_DATOS / "division.json"))
for k, v in division.items():
    print(f"{k:14s}: {len(v)} videos  {v}")

In [ ]:
video = division["prueba"][0]
rgb, prof, mask = c3vd.cargar_video(video)
i = len(rgb) // 2
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].imshow(rgb[i]); ax[0].set_title(f"{video} · frame {i}")
im = ax[1].imshow(np.where(prof[i] > 0, prof[i], np.nan), cmap="magma_r"); ax[1].set_title("profundidad real (mm)")
plt.colorbar(im, ax=ax[1], fraction=0.04)
ax[2].imshow(rgb[i]); ax[2].imshow(np.ma.masked_where(~mask[i], mask[i]), cmap="autumn", alpha=0.7)
ax[2].set_title(f"reflejos ({100 * mask[i].mean():.1f}% del frame)")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

## 2. Endo-STTN congelado borra los reflejos

Endo-STTN no se entrena: se usa el modelo preentrenado (`gen_00009.pth`) con la inferencia oficial, que busca en
frames vecinos y de referencia el tejido que el reflejo oculta. Fuera de la máscara el frame no cambia.

In [ ]:
if CORRER_LIMPIEZA:
    correr("codigo/limpiar_endosttn.py")

limpio = c3vd.cargar_video(video, "rgb_limpio")[0]
ys, xs = np.nonzero(mask[i]) if mask[i].any() else (np.array([128]), np.array([160]))
cy, cx = int(np.median(ys)), int(np.median(xs))
y0, x0 = np.clip(cy - 48, 0, rgb.shape[1] - 96), np.clip(cx - 60, 0, rgb.shape[2] - 120)
recorte = lambda a: a[y0:y0 + 96, x0:x0 + 120]
fig, ax = plt.subplots(1, 3, figsize=(13, 4))
for a, img, t in zip(ax, [rgb[i], limpio[i], recorte(limpio[i])], ["original", "limpio (Endo-STTN)", "zoom en el reflejo (limpio)"]):
    a.imshow(img); a.set_title(t); a.axis("off")
ax[0].add_patch(plt.Rectangle((x0, y0), 120, 96, fill=False, color="cyan"))
plt.tight_layout(); plt.show()
print("Píxeles cambiados fuera de la máscara:", int((np.abs(limpio[i].astype(int) - rgb[i]).sum(-1)[~mask[i]] > 0).sum()))

## 3. Ajuste fino del estimador de profundidad

Se entrenan **dos** estimadores desde los mismos pesos (AF-SfMLearner, SCARED) y con la misma configuración:
**C** con frames originales y **D** con frames limpios. Pérdida *scale-invariant log* contra la profundidad real;
se guarda la época con menor AbsRel de validación.

In [ ]:
if CORRER_AJUSTE:
    for entrada in ["rgb", "rgb_limpio"]:
        correr("codigo/ajustar_profundidad.py", "--entrada", entrada, "--epocas", EPOCAS, "--semilla", SEMILLA)

fig, ax = plt.subplots(figsize=(7, 4))
for entrada, nombre in [("rgb", "C · originales"), ("rgb_limpio", "D · limpios")]:
    h = c3vd.DIR_MODELOS / f"{entrada}_s{SEMILLA}" / "historial.json"
    if h.exists():
        ep = json.load(open(h))["epocas"]
        ax.plot([e["epoca"] for e in ep], [e["abs_rel_val"] for e in ep], "o-", label=nombre)
ax.set_xlabel("época"); ax.set_ylabel("AbsRel validación"); ax.grid(alpha=.3); ax.legend()
plt.tight_layout(); plt.show()

## 4. Evaluación en videos no vistos

| Modelo | Pesos | Frames de prueba |
|---|---|---|
| A | AF-SfMLearner (SCARED) | originales |
| B | AF-SfMLearner (SCARED) | limpios |
| C | ajustado con originales | originales |
| D | ajustado con limpios | limpios |

Métricas con *median scaling* (escala calculada con toda la imagen), en tres regiones: toda la imagen,
dentro de los reflejos y un anillo de 10 px alrededor. Wilcoxon pareado por frame.

In [ ]:
if CORRER_EVALUAR:
    correr("codigo/evaluar_profundidad.py", "--semilla", SEMILLA)
from IPython.display import Markdown, display
display(Markdown(open(c3vd.DIR_RESULTADOS / "evaluacion_resumen.md").read()))

In [ ]:
modelos = {"A": (mp.PESOS_SCARED, "rgb"), "B": (mp.PESOS_SCARED, "rgb_limpio"),
           "C": (c3vd.DIR_MODELOS / f"rgb_s{SEMILLA}", "rgb"), "D": (c3vd.DIR_MODELOS / f"rgb_limpio_s{SEMILLA}", "rgb_limpio")}
g = np.asarray(prof[i], np.float32); valido = g > 0
fig, ax = plt.subplots(2, 5, figsize=(20, 7))
ax[0, 0].imshow(rgb[i]); ax[0, 0].set_title("frame original")
ax[1, 0].imshow(np.where(valido, g, np.nan), cmap="magma_r"); ax[1, 0].set_title("profundidad real")
for j, (nombre, (pesos, entrada)) in enumerate(modelos.items(), 1):
    if not (Path(pesos) / "encoder.pth").exists():
        ax[0, j].set_title(f"{nombre}: sin modelo"); continue
    enc, dec = mp.cargar(pesos, DEVICE); enc.eval(); dec.eval()
    x = c3vd.cargar_video(video, entrada)[0][i:i + 1]
    with torch.no_grad():
        p = mp.predecir(enc, dec, mp.a_tensor(x, DEVICE))[0, 0].cpu().numpy()
    p = p * np.median(g[valido]) / np.median(p[valido])
    err = np.where(valido, np.abs(p - g) / np.maximum(g, 1e-3), np.nan)
    ax[0, j].imshow(p, cmap="magma_r"); ax[0, j].set_title(f"{nombre}: predicción")
    ax[1, j].imshow(err, cmap="inferno", vmin=0, vmax=0.5); ax[1, j].set_title(f"{nombre}: error relativo")
    ax[1, j].contour(mask[i], [0.5], colors="cyan", linewidths=0.6)
for a in ax.ravel(): a.axis("off")
plt.tight_layout(); plt.show()

## 5. Cómo leer los resultados

- **D < C en AbsRel dentro de los reflejos** (Wilcoxon p < 0.05) y sin empeorar en toda la imagen: entrenar con frames
  limpios mejora la profundidad donde había reflejos. Es el resultado buscado.
- **B < A**: limpiar ayuda incluso sin reentrenar el estimador.
- **C ≈ D**: el estimador ya tolera los reflejos de C3VDv2, o la limpieza no aporta información útil.
- Si D mejora en la imagen pero no en los reflejos, la ganancia viene de otro lado (p. ej., estabilidad del entrenamiento).

Repite con otras semillas (`SEMILLA = 1, 2`) para comprobar que la diferencia no es casualidad.